# TEM-Inspired World Model Training
**Runtime → Change runtime type → A100 GPU** before running.

In [ ]:
# Setup
!git clone https://github.com/YX234/tem-generalization.git
%cd tem-generalization
!pip install -q gymnasium[mujoco] mujoco scikit-learn

In [ ]:
# Verify GPU
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Create output directory on Drive
import os
DRIVE_DIR = '/content/drive/MyDrive/tem-generalization'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive output: {DRIVE_DIR}')


In [ ]:
# Train (checkpoints auto-synced to Drive every 1000 iterations)
import os
os.environ['TEM_DRIVE_DIR'] = DRIVE_DIR
!python train.py


---
## Phase 2: RL Policy Training
Train a PPO policy on the frozen TEM representations.
The policy receives g_inf (54-dim abstract state) and learns to produce Hopper actions.

**Prerequisites:** World model training (Phase 1) must be complete.

In [ ]:
# Find the latest world model checkpoint
import glob, os

# Check Google Drive first, then local
drive_models = os.path.join(DRIVE_DIR, 'models') if 'DRIVE_DIR' in dir() else ''
local_models = sorted(glob.glob('runs/*/models'))

if os.path.isdir(drive_models) and glob.glob(os.path.join(drive_models, 'tem_*.pt')):
    model_dir = drive_models
    print(f'Using Drive checkpoint: {model_dir}')
elif local_models:
    model_dir = local_models[-1]
    print(f'Using local checkpoint: {model_dir}')
else:
    raise FileNotFoundError('No world model checkpoint found. Run Phase 1 first.')

# List available checkpoints
ckpts = sorted(glob.glob(os.path.join(model_dir, 'tem_*.pt')))
print(f'Available checkpoints: {[os.path.basename(c) for c in ckpts]}')


In [ ]:
# Train PPO policy on TEM representations (1M timesteps, ~30-60 min)
# Checkpoints auto-synced to Google Drive every 10 updates
import os
os.environ['TEM_DRIVE_DIR'] = DRIVE_DIR
!python train_rl.py --model-dir {model_dir}


### RL Evaluation
Test the trained policy across domain-randomized environments.
This measures whether g_inf is sufficient for cross-environment control.

In [ ]:
# Evaluate trained policy
import glob, os, torch, numpy as np
from config import make_config, make_rl_config
from tem_model import TEMModel
from tem_wrapper import TEMObservationEnv
from policy import ActorCritic
from environment import RunningNormalizer
import pickle

# Load policy
rl_runs = sorted(glob.glob('runs_rl/*'))
if not rl_runs:
    raise FileNotFoundError('No RL runs found. Train the policy first.')
rl_run_dir = rl_runs[-1]
policy_path = os.path.join(rl_run_dir, 'policy_final.pt')
if not os.path.exists(policy_path):
    # Use latest numbered checkpoint
    policy_files = sorted(glob.glob(os.path.join(rl_run_dir, 'policy_*.pt')))
    policy_path = policy_files[-1]
print(f'Policy: {policy_path}')

# Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tem_cfg = torch.load(os.path.join(model_dir, 'cfg_final.pt'), weights_only=False)
tem = TEMModel(tem_cfg, device=device).to(device)
tem.load_state_dict(torch.load(os.path.join(model_dir, 'tem_final.pt'), map_location=device, weights_only=True))
tem.eval()
for p in tem.parameters():
    p.requires_grad_(False)

with open(os.path.join(model_dir, 'normalizer_final.pkl'), 'rb') as f:
    normalizer = pickle.load(f)

g_dim = sum(tem_cfg['n_g'])
rl_cfg = make_rl_config()
policy = ActorCritic(g_dim, tem_cfg['action_dim'], rl_cfg['hidden_dim']).to(device)
ckpt = torch.load(policy_path, map_location=device, weights_only=False)
policy.load_state_dict(ckpt['policy_state_dict'])
policy.eval()
print(f'Loaded policy from update {ckpt["update"]}, step {ckpt["global_step"]}')

# Test environments
test_configs = {
    'Default': {'randomize': False, 'gravity_range': None, 'terminate_when_unhealthy': True},
    'In-Distribution': {'terminate_when_unhealthy': True},
    'OOD Extrapolation': {
        'terminate_when_unhealthy': True,
        'gravity_range': tem_cfg.get('eval_gravity_range', (2.5, 4.0)),
        'friction_range': tem_cfg.get('eval_friction_range', (0.02, 0.1)),
        'mass_range': tem_cfg.get('eval_mass_range', (4.0, 8.0)),
    },
    'Extreme Gravity': {
        'terminate_when_unhealthy': True,
        'mass_range': (1.0, 1.0), 'damping_range': (1.0, 1.0),
        'friction_range': (1.0, 1.0), 'gear_range': (1.0, 1.0),
        'gravity_range': (3.0, 4.0),
    },
}

n_eval = 20
max_steps = 1000
results = {}

for name, overrides in test_configs.items():
    env_cfg = {**tem_cfg, **overrides}
    rewards, lengths = [], []
    for ep in range(n_eval):
        env = TEMObservationEnv(env_cfg, tem, normalizer, device, seed=5000+ep)
        obs, _ = env.reset()
        total_reward = 0
        for step in range(max_steps):
            with torch.no_grad():
                obs_t = torch.tensor(obs, dtype=torch.float, device=device).unsqueeze(0)
                action, _, _, _ = policy(obs_t)
                action = action.squeeze(0).cpu().numpy()
            obs, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            if terminated or truncated:
                break
        rewards.append(total_reward)
        lengths.append(step + 1)
        env.close()
    results[name] = {'rewards': rewards, 'lengths': lengths}
    print(f'{name}: reward={np.mean(rewards):.1f} +/- {np.std(rewards):.1f}, '
          f'length={np.mean(lengths):.0f} +/- {np.std(lengths):.0f}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
names = list(results.keys())
colors = ['#2196F3', '#4CAF50', '#FF5722', '#9C27B0']

ax = axes[0]
for i, name in enumerate(names):
    r = results[name]['rewards']
    ax.bar(i, np.mean(r), yerr=np.std(r), color=colors[i], alpha=0.8, capsize=5)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylabel('Episode Reward')
ax.set_title('Cross-Environment Policy Performance')
ax.grid(True, alpha=0.3)

ax = axes[1]
for i, name in enumerate(names):
    l = results[name]['lengths']
    ax.bar(i, np.mean(l), yerr=np.std(l), color=colors[i], alpha=0.8, capsize=5)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylabel('Episode Length')
ax.set_title('Survival Duration (longer = better adaptation)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
## Visualizations
Run these cells after training completes to inspect the learned representations.

In [ ]:
# Load trained model and normalizer
import glob, os, sys, pickle
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

sys.path.insert(0, '.')
from tem_model import TEMModel
from environment import DomainRandomizedHopper, RunningNormalizer

# Find latest checkpoint
model_dirs = sorted(glob.glob('runs/*/models'))
if not model_dirs:
    raise FileNotFoundError('No training runs found. Train the model first.')
model_dir = model_dirs[-1]
# Use final or latest numbered checkpoint
ckpt = os.path.join(model_dir, 'tem_final.pt')
if not os.path.exists(ckpt):
    ckpts = sorted(glob.glob(os.path.join(model_dir, 'tem_*.pt')))
    ckpt = ckpts[-1]
print(f'Loading: {ckpt}')

# Load saved config from checkpoint (must match the model architecture)
idx = os.path.basename(ckpt).replace('tem_', '').replace('.pt', '')
cfg_path = os.path.join(model_dir, f'cfg_{idx}.pt')
cfg = torch.load(cfg_path, weights_only=False)
print(f'Config loaded: g_dim={sum(cfg["n_g"])}, n_p={sum(cfg["n_p"])}')

# Load normalizer (must match the one used during training)
norm_path = os.path.join(model_dir, f'normalizer_{idx}.pkl')
if not os.path.exists(norm_path):
    norm_paths = sorted(glob.glob(os.path.join(model_dir, 'normalizer_*.pkl')))
    norm_path = norm_paths[-1] if norm_paths else None
if norm_path:
    with open(norm_path, 'rb') as f:
        normalizer = pickle.load(f)
    print(f'Normalizer loaded (trained on {normalizer.count} observations)')
else:
    print('WARNING: No normalizer found — using raw observations (results may be wrong)')
    normalizer = None

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TEMModel(cfg, device=device).to(device)
model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
model.eval()
print(f'Model loaded on {device}')

def normalize_obs(obs_raw):
    """Normalize a single observation using the training normalizer."""
    if normalizer is not None:
        return normalizer.normalize(obs_raw)
    return obs_raw.astype(np.float32)

In [ ]:
# Collect representations from multiple domain-randomized episodes
n_episodes = 8
steps_per_ep = 200

all_g, all_obs, all_obs_raw, all_pred, all_ep, all_t, all_masses = [], [], [], [], [], [], []

for ep in range(n_episodes):
    cfg_eval = cfg.copy()
    cfg_eval['randomize'] = True
    env = DomainRandomizedHopper(cfg_eval, seed=5000 + ep)
    obs, _ = env.reset()
    total_mass = float(env.body_params['body_mass'].sum())

    chunk = []
    raw_obs_list = []
    for t in range(steps_per_ep):
        action = env.action_space.sample()
        obs_normed = normalize_obs(obs)
        raw_obs_list.append(obs.copy())
        chunk.append({
            'obs': torch.tensor(obs_normed, dtype=torch.float).unsqueeze(0).to(device),
            'action': torch.tensor(action, dtype=torch.float).unsqueeze(0).to(device),
        })
        obs, _, term, trunc, _ = env.step(action)
        if term or trunc:
            obs, _ = env.reset()
    env.close()

    with torch.no_grad():
        steps = model(chunk)

    for t, step in enumerate(steps):
        all_g.append(torch.cat(step.g_inf, dim=1).squeeze(0).cpu().numpy())
        all_obs.append(step.obs.squeeze(0).cpu().numpy())  # normalized
        all_obs_raw.append(raw_obs_list[t])
        all_pred.append(step.x_gen[0][0].squeeze(0).cpu().numpy())  # prediction (normalized space)
        all_ep.append(ep)
        all_t.append(t)
        all_masses.append(total_mass)

all_g = np.stack(all_g)
all_obs = np.stack(all_obs)
all_obs_raw = np.stack(all_obs_raw)
all_pred = np.stack(all_pred)
all_ep = np.array(all_ep)
all_t = np.array(all_t)
all_masses = np.array(all_masses)

print(f'Collected {len(all_g)} data points from {n_episodes} episodes')

### 1. Training Loss Curves

In [ ]:
# Parse training log for loss curves
import re

log_files = sorted(glob.glob('runs/*/train.log'))
if not log_files:
    print('No log file found')
else:
    iters, losses, mse_p, mse_ginf, mse_ggen = [], [], [], [], []
    with open(log_files[-1]) as f:
        for line in f:
            m = re.search(r'Iter\s+(\d+).*loss ([\-\d.]+).*mse\(p\) ([\d.]+) mse\(g_inf\) ([\d.]+) mse\(g_gen\) ([\d.]+)', line)
            if m:
                iters.append(int(m.group(1)))
                losses.append(float(m.group(2)))
                mse_p.append(float(m.group(3)))
                mse_ginf.append(float(m.group(4)))
                mse_ggen.append(float(m.group(5)))

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    ax = axes[0]
    ax.plot(iters, losses, alpha=0.3, color='blue')
    # Smoothed
    if len(losses) > 50:
        w = 50
        smoothed = np.convolve(losses, np.ones(w)/w, mode='valid')
        ax.plot(iters[w-1:], smoothed, color='blue', linewidth=2, label='smoothed')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Total Loss')
    ax.set_title('Training Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(iters, mse_p, alpha=0.3, color='red')
    ax.plot(iters, mse_ginf, alpha=0.3, color='green')
    ax.plot(iters, mse_ggen, alpha=0.3, color='blue')
    if len(mse_p) > 50:
        for vals, color, label in [(mse_p, 'red', 'from p_inf'), (mse_ginf, 'green', 'from g_inf'), (mse_ggen, 'blue', 'from g_gen')]:
            smoothed = np.convolve(vals, np.ones(w)/w, mode='valid')
            ax.plot(iters[w-1:], smoothed, color=color, linewidth=2, label=label)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Observation MSE')
    ax.set_title('Observation Prediction Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    print(f'Final obs MSE: {mse_p[-1]:.4f} (from p), {mse_ginf[-1]:.4f} (from g_inf), {mse_ggen[-1]:.4f} (from g_gen)')

### 2. Observation Prediction Quality
How well does the model predict each observation dimension?

In [ ]:
dim_names = ['z_pos', 'angle', 'thigh_ang', 'leg_ang', 'foot_ang',
             'x_vel', 'z_vel', 'ang_vel', 'thigh_vel', 'leg_vel', 'foot_vel']

# MSE in normalized space (what the model optimizes)
mse_per_dim = np.mean((all_obs - all_pred) ** 2, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Bar chart of per-dimension MSE (normalized space)
ax = axes[0]
colors = ['#2196F3' if mse < np.median(mse_per_dim) else '#FF5722' for mse in mse_per_dim]
ax.bar(range(len(dim_names)), mse_per_dim, color=colors)
ax.set_xticks(range(len(dim_names)))
ax.set_xticklabels(dim_names, rotation=45, ha='right')
ax.set_ylabel('MSE (normalized)')
ax.set_title(f'Prediction MSE per Dimension (total={np.mean(mse_per_dim):.4f})')
ax.grid(True, alpha=0.3, axis='y')

# Predicted vs actual scatter for best and worst dimensions
ax = axes[1]
best_dim = np.argmin(mse_per_dim)
worst_dim = np.argmax(mse_per_dim)
subset = np.random.choice(len(all_obs), min(500, len(all_obs)), replace=False)
ax.scatter(all_obs[subset, best_dim], all_pred[subset, best_dim], s=3, alpha=0.4, label=f'Best: {dim_names[best_dim]}')
ax.scatter(all_obs[subset, worst_dim], all_pred[subset, worst_dim], s=3, alpha=0.4, label=f'Worst: {dim_names[worst_dim]}')
lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
ax.plot(lims, lims, 'k--', alpha=0.3, label='Perfect')
ax.set_xlabel('Actual (normalized)')
ax.set_ylabel('Predicted (normalized)')
ax.set_title('Predicted vs Actual Observations')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3. Abstract State Space (g) Visualization
Does g capture dynamical structure (gait phase, balance) rather than body-specific details?

In [ ]:
# t-SNE of g-space
print('Running t-SNE (may take a minute)...')
tsne = TSNE(n_components=2, perplexity=30, random_state=0)
g_2d = tsne.fit_transform(all_g)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Color by episode (different body params)
ax = axes[0, 0]
scatter = ax.scatter(g_2d[:, 0], g_2d[:, 1], c=all_ep, cmap='tab10', s=3, alpha=0.5)
plt.colorbar(scatter, ax=ax, label='Episode')
ax.set_title('g-space by episode (different bodies)')

# Color by time step
ax = axes[0, 1]
scatter = ax.scatter(g_2d[:, 0], g_2d[:, 1], c=all_t, cmap='viridis', s=3, alpha=0.5)
plt.colorbar(scatter, ax=ax, label='Time step')
ax.set_title('g-space by time within episode')

# Color by body height (z_pos) — a key dynamical variable
ax = axes[1, 0]
scatter = ax.scatter(g_2d[:, 0], g_2d[:, 1], c=all_obs[:, 0], cmap='coolwarm', s=3, alpha=0.5)
plt.colorbar(scatter, ax=ax, label='z_pos (height)')
ax.set_title('g-space by body height')

# Color by total body mass — if g captures structure, mass shouldn't cluster
ax = axes[1, 1]
scatter = ax.scatter(g_2d[:, 0], g_2d[:, 1], c=all_masses, cmap='plasma', s=3, alpha=0.5)
plt.colorbar(scatter, ax=ax, label='Total mass')
ax.set_title('g-space by body mass\n(overlap = structural abstraction)')

for ax in axes.flat:
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')

plt.tight_layout()
plt.show()

### 4. Frequency Module Analysis
Do different frequency modules capture different timescales?

In [ ]:
# Collect single long episode for temporal analysis
cfg_fixed = cfg.copy()
cfg_fixed['randomize'] = False
env = DomainRandomizedHopper(cfg_fixed, seed=42)
obs, _ = env.reset()

n_steps_long = 400
chunk = []
for t in range(n_steps_long):
    action = env.action_space.sample()
    obs_normed = normalize_obs(obs)
    chunk.append({
        'obs': torch.tensor(obs_normed, dtype=torch.float).unsqueeze(0).to(device),
        'action': torch.tensor(action, dtype=torch.float).unsqueeze(0).to(device),
    })
    obs, _, term, trunc, _ = env.step(action)
    if term or trunc:
        obs, _ = env.reset()
env.close()

with torch.no_grad():
    steps_long = model(chunk)

n_f = cfg['n_f']
fig, axes = plt.subplots(n_f, 2, figsize=(14, 3.5 * n_f))

for f in range(n_f):
    g_f = np.stack([s.g_inf[f].squeeze(0).cpu().numpy() for s in steps_long])

    # Activations over time
    ax = axes[f, 0]
    n_show = min(6, g_f.shape[1])
    for neuron in range(n_show):
        ax.plot(g_f[:, neuron], alpha=0.7, linewidth=0.8)
    ax.set_title(f'Module {f} (alpha={cfg["f_initial"][f]:.2f}, n_g={cfg["n_g"][f]}): activations')
    ax.set_xlabel('Step')
    ax.set_ylabel('Activation')
    ax.grid(True, alpha=0.2)

    # Autocorrelation — should decay faster for high-freq modules
    ax = axes[f, 1]
    max_lag = min(100, len(g_f) // 2)
    mean_autocorr = np.zeros(max_lag)
    n_valid = 0
    for neuron in range(g_f.shape[1]):
        signal = g_f[:, neuron] - np.mean(g_f[:, neuron])
        norm = np.sum(signal ** 2)
        if norm > 1e-10:
            autocorr = np.correlate(signal, signal, mode='full')
            autocorr = autocorr[len(autocorr)//2 : len(autocorr)//2 + max_lag]
            mean_autocorr += autocorr / norm
            n_valid += 1
    if n_valid > 0:
        mean_autocorr /= n_valid
    ax.plot(mean_autocorr, color='steelblue', linewidth=2)
    ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
    ax.set_title(f'Module {f}: autocorrelation (slow decay = slow dynamics)')
    ax.set_xlabel('Lag (steps)')
    ax.set_ylabel('Autocorrelation')
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

### 5. Cross-Domain Transfer
Do similar dynamical states map to similar g-vectors across different body configurations?

In [ ]:
# PCA of g-space across 6 different body configs
n_configs = 6
config_g, config_height, config_angle, config_id = [], [], [], []

for c in range(n_configs):
    cfg_c = cfg.copy()
    cfg_c['randomize'] = True
    env = DomainRandomizedHopper(cfg_c, seed=9000 + c)
    obs, _ = env.reset()

    chunk = []
    raw_obs_list = []
    for t in range(200):
        action = env.action_space.sample()
        obs_normed = normalize_obs(obs)
        raw_obs_list.append(obs.copy())
        chunk.append({
            'obs': torch.tensor(obs_normed, dtype=torch.float).unsqueeze(0).to(device),
            'action': torch.tensor(action, dtype=torch.float).unsqueeze(0).to(device),
        })
        obs, _, term, trunc, _ = env.step(action)
        if term or trunc:
            obs, _ = env.reset()
    env.close()

    with torch.no_grad():
        steps_c = model(chunk)
    for i, step in enumerate(steps_c):
        config_g.append(torch.cat(step.g_inf, dim=1).squeeze(0).cpu().numpy())
        config_height.append(raw_obs_list[i][0])   # raw z_pos for interpretable axis
        config_angle.append(raw_obs_list[i][1])     # raw angle
        config_id.append(c)

config_g = np.stack(config_g)
config_height = np.array(config_height)
config_angle = np.array(config_angle)
config_id = np.array(config_id)

pca = PCA(n_components=2)
g_pca = pca.fit_transform(config_g)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
for c in range(n_configs):
    mask = config_id == c
    ax.scatter(g_pca[mask, 0], g_pca[mask, 1], s=5, alpha=0.4, label=f'Body {c}')
ax.set_title('g-space: different body configs\n(overlap = generalization)')
ax.legend(fontsize=8, markerscale=3)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')

ax = axes[1]
scatter = ax.scatter(g_pca[:, 0], g_pca[:, 1], c=config_height, cmap='coolwarm', s=5, alpha=0.4)
plt.colorbar(scatter, ax=ax, label='Body height (z)')
ax.set_title('g-space colored by height\n(smooth gradient = structural encoding)')
ax.set_xlabel(f'PC1')
ax.set_ylabel(f'PC2')

ax = axes[2]
scatter = ax.scatter(g_pca[:, 0], g_pca[:, 1], c=config_angle, cmap='RdYlGn', s=5, alpha=0.4)
plt.colorbar(scatter, ax=ax, label='Body angle')
ax.set_title('g-space colored by body angle\n(smooth gradient = structural encoding)')
ax.set_xlabel(f'PC1')
ax.set_ylabel(f'PC2')

plt.tight_layout()
plt.show()

print(f'PCA explained variance: PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}')

### 6. Prediction Traces
Overlay predicted vs actual observations over time for a single episode.

In [ ]:
# Use the long episode from section 4
# Predictions and observations are in normalized space
obs_trace = np.stack([s.obs.squeeze(0).cpu().numpy() for s in steps_long])
pred_trace = np.stack([s.x_gen[0][0].squeeze(0).cpu().numpy() for s in steps_long])

dims_to_plot = [0, 1, 5, 7]  # z_pos, angle, x_vel, ang_vel
fig, axes = plt.subplots(len(dims_to_plot), 1, figsize=(14, 2.5 * len(dims_to_plot)), sharex=True)

for i, d in enumerate(dims_to_plot):
    ax = axes[i]
    ax.plot(obs_trace[:, d], color='black', linewidth=1, label='Actual', alpha=0.8)
    ax.plot(pred_trace[:, d], color='red', linewidth=1, label='Predicted', alpha=0.7, linestyle='--')
    ax.set_ylabel(f'{dim_names[d]} (norm.)')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.2)
    mse = np.mean((obs_trace[:, d] - pred_trace[:, d]) ** 2)
    ax.set_title(f'{dim_names[d]} (MSE={mse:.4f})', fontsize=10)

axes[-1].set_xlabel('Step')
plt.suptitle('Predicted vs Actual Observations Over Time (normalized space)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

### 7. Episodic Memory Buffer
How does the episodic buffer fill during an episode?

In [ ]:
# Track episodic buffer fill and key diversity over time
buf_counts = [s.M[0]['count'].item() for s in steps_long]
K = cfg.get('episodic_capacity', 50)

# Compute key diversity: mean pairwise cosine distance among stored keys
key_diversity = []
for s in steps_long:
    buf = s.M[0]
    n = buf['count'].item()
    if n < 2:
        key_diversity.append(0.0)
    else:
        keys = buf['keys'][0, :n]  # (n, n_p_total)
        keys_norm = torch.nn.functional.normalize(keys, dim=1)
        sim = torch.mm(keys_norm, keys_norm.t())  # (n, n)
        # Mean off-diagonal cosine distance
        mask = ~torch.eye(n, dtype=torch.bool, device=sim.device)
        key_diversity.append((1 - sim[mask].mean()).cpu().item())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(buf_counts, color='steelblue', linewidth=1.5)
ax.axhline(y=K, color='coral', linestyle='--', alpha=0.7, label=f'Capacity (K={K})')
ax.set_xlabel('Step')
ax.set_ylabel('Entries stored')
ax.set_title('Episodic Buffer Fill Level')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(key_diversity, color='coral', linewidth=1.5)
ax.set_xlabel('Step')
ax.set_ylabel('Mean cosine distance')
ax.set_title('Key Diversity (higher = more distinct memories)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 8. Grid Cell Analysis
In the original TEM for 2D navigation, individual g neurons develop hexagonal firing fields (grid cells). In the Hopper's dynamical state space, the analogue is **structured firing fields** — individual g neurons that activate selectively for specific regions of the state space (e.g., body height x angle).

We look for three signatures:
1. **2D firing rate maps**: g-neuron activation binned over state-variable pairs — structured blobs rather than random noise
2. **1D tuning curves**: smooth, selective responses to individual state variables — different widths across frequency modules (multi-scale)
3. **Gridness score**: periodic structure in the spatial autocorrelation of firing maps (standard neuroscience metric; >0.37 = grid cell)

In [ ]:
# Collect dense state-space data for grid cell analysis
# Long episodes (termination disabled) give much better state-space coverage
from scipy.ndimage import gaussian_filter

n_ep_grid = 12
n_steps_grid = 500

grid_g_modules = [[] for _ in range(cfg['n_f'])]
grid_raw = []

print(f'Collecting {n_ep_grid} x {n_steps_grid} = {n_ep_grid * n_steps_grid} data points...')
for ep in range(n_ep_grid):
    cfg_g = cfg.copy()
    cfg_g['randomize'] = True
    env = DomainRandomizedHopper(cfg_g, seed=7000 + ep)
    obs, _ = env.reset()

    chunk, raw_list = [], []
    for t in range(n_steps_grid):
        action = env.action_space.sample()
        raw_list.append(obs.copy())
        obs_normed = normalize_obs(obs)
        chunk.append({
            'obs': torch.tensor(obs_normed, dtype=torch.float).unsqueeze(0).to(device),
            'action': torch.tensor(action, dtype=torch.float).unsqueeze(0).to(device),
        })
        obs, _, term, trunc, _ = env.step(action)
        if term or trunc:
            obs, _ = env.reset()
    env.close()

    with torch.no_grad():
        steps_g = model(chunk)
    for t, step in enumerate(steps_g):
        for f in range(cfg['n_f']):
            grid_g_modules[f].append(step.g_inf[f].squeeze(0).cpu().numpy())
        grid_raw.append(raw_list[t])

grid_g_modules = [np.stack(g) for g in grid_g_modules]
grid_raw = np.stack(grid_raw)
n_f = cfg['n_f']
print(f'Collected: {grid_raw.shape[0]} points, modules: {[g.shape for g in grid_g_modules]}')

# Helper: compute 2D firing rate map for a neuron
def compute_firing_map(activations, var1, var2, n_bins=25):
    v1_lo, v1_hi = np.percentile(var1, [2, 98])
    v2_lo, v2_hi = np.percentile(var2, [2, 98])
    mask = (var1 >= v1_lo) & (var1 <= v1_hi) & (var2 >= v2_lo) & (var2 <= v2_hi)
    v1_bins = np.linspace(v1_lo, v1_hi, n_bins + 1)
    v2_bins = np.linspace(v2_lo, v2_hi, n_bins + 1)
    rate_map = np.full((n_bins, n_bins), np.nan)
    for i in range(n_bins):
        for j in range(n_bins):
            in_bin = mask & (var1 >= v1_bins[i]) & (var1 < v1_bins[i+1]) & \
                     (var2 >= v2_bins[j]) & (var2 < v2_bins[j+1])
            if np.sum(in_bin) >= 3:
                rate_map[j, i] = np.mean(activations[in_bin])
    # Smooth, handling NaNs
    valid = ~np.isnan(rate_map)
    rm = rate_map.copy(); rm[~valid] = 0
    rm = gaussian_filter(rm, sigma=1.0)
    cnt = gaussian_filter(valid.astype(float), sigma=1.0)
    rm[cnt > 0.1] /= cnt[cnt > 0.1]
    rm[cnt <= 0.1] = np.nan
    return rm, [v1_lo, v1_hi, v2_lo, v2_hi]

# 2D firing rate maps for the 3 most variable neurons per module
# Two state-space planes: (z_pos, angle) and (z_pos, z_vel)
planes = [
    (0, 1, 'z_pos', 'angle'),
    (0, 6, 'z_pos', 'z_vel'),
]

for var1_idx, var2_idx, v1_name, v2_name in planes:
    fig, axes = plt.subplots(n_f, 3, figsize=(12, 3 * n_f))
    fig.suptitle(f'2D Firing Rate Maps: g-neuron activation on ({v1_name}, {v2_name}) plane\n'
                 'Structured patterns = grid-cell-like spatial encoding', fontsize=12, y=1.02)

    for f in range(n_f):
        g_mod = grid_g_modules[f]
        # 3 most variable neurons
        var_per_neuron = np.var(g_mod, axis=0)
        top3 = np.argsort(var_per_neuron)[-3:][::-1]

        for col, neuron in enumerate(top3):
            ax = axes[f, col]
            rm, extent = compute_firing_map(
                g_mod[:, neuron], grid_raw[:, var1_idx], grid_raw[:, var2_idx]
            )
            im = ax.imshow(rm, origin='lower', aspect='auto', cmap='hot',
                           extent=extent, interpolation='bilinear')
            plt.colorbar(im, ax=ax, shrink=0.8)
            ax.set_title(f'Mod {f} neuron {neuron}', fontsize=9)
            if col == 0:
                ax.set_ylabel(f'Module {f}\n{v2_name}')
            if f == n_f - 1:
                ax.set_xlabel(v1_name)

    plt.tight_layout()
    plt.show()

In [ ]:
# 1D tuning curves: how does each g neuron respond to individual state variables?
# Different frequency modules should show tuning at different scales (bandwidths).
state_vars = [0, 1, 2, 5, 7]  # z_pos, angle, thigh_ang, x_vel, ang_vel
state_names = ['z_pos', 'angle', 'thigh_ang', 'x_vel', 'ang_vel']

fig, axes = plt.subplots(n_f, len(state_vars), figsize=(3.5 * len(state_vars), 3 * n_f))

for f in range(n_f):
    g_mod = grid_g_modules[f]
    # Pick 3 most variable neurons
    var_per_neuron = np.var(g_mod, axis=0)
    top_neurons = np.argsort(var_per_neuron)[-3:][::-1]

    for vi, (var_idx, var_name) in enumerate(zip(state_vars, state_names)):
        ax = axes[f, vi]
        sv = grid_raw[:, var_idx]
        lo, hi = np.percentile(sv, [2, 98])
        bins = np.linspace(lo, hi, 30)
        bin_centers = 0.5 * (bins[:-1] + bins[1:])
        bin_idx = np.digitize(sv, bins) - 1

        for rank, neuron in enumerate(top_neurons):
            means = []
            for b in range(len(bins) - 1):
                mask = bin_idx == b
                means.append(np.mean(g_mod[mask, neuron]) if mask.sum() >= 5 else np.nan)
            ax.plot(bin_centers, means, linewidth=1.5, alpha=0.8, label=f'g[{neuron}]')

        ax.set_xlabel(var_name, fontsize=9)
        if vi == 0:
            ax.set_ylabel(f'Module {f}\n(alpha={cfg["f_initial"][f]:.2f})', fontsize=9)
        if f == 0:
            ax.set_title(var_name, fontsize=10)
        ax.grid(True, alpha=0.2)
        if vi == len(state_vars) - 1 and f == 0:
            ax.legend(fontsize=6, loc='upper right')

plt.suptitle('1D Tuning Curves: g-Neuron Activation vs State Variables\n'
             'Slow modules (bottom) should show broader, smoother tuning than fast modules (top)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

# Quantify tuning width per module
print('Mean tuning width (std of activation-weighted state value):')
for f in range(n_f):
    g_mod = grid_g_modules[f]
    widths = []
    for neuron in range(g_mod.shape[1]):
        act = g_mod[:, neuron]
        act_pos = np.maximum(act - np.median(act), 0)
        if act_pos.sum() > 0:
            weights = act_pos / act_pos.sum()
            weighted_mean = np.sum(weights * grid_raw[:, 0])
            weighted_std = np.sqrt(np.sum(weights * (grid_raw[:, 0] - weighted_mean) ** 2))
            widths.append(weighted_std)
    print(f'  Module {f} (alpha={cfg["f_initial"][f]:.2f}): {np.mean(widths):.3f} '
          f'(expect: slow modules = wider tuning)')

In [ ]:
# Gridness score: does the spatial autocorrelation of firing maps show periodic structure?
# In neuroscience, gridness > 0.37 indicates a grid cell. In state-space, we look for
# any structured periodicity (not necessarily hexagonal).
from scipy.signal import correlate2d
from scipy.ndimage import rotate as ndimage_rotate

def compute_gridness(rate_map):
    """Gridness score from 2D spatial autocorrelation."""
    rm = rate_map.copy()
    rm[np.isnan(rm)] = 0
    rm -= np.mean(rm)
    if np.std(rm) < 1e-10:
        return 0.0, np.zeros_like(rm), {}
    autocorr = correlate2d(rm, rm, mode='same')
    autocorr /= np.max(np.abs(autocorr)) + 1e-10
    center = np.array(autocorr.shape) // 2
    r_max = min(center) * 0.75
    y, x = np.ogrid[:autocorr.shape[0], :autocorr.shape[1]]
    dist = np.sqrt((x - center[1])**2 + (y - center[0])**2)
    ring = (dist > 3) & (dist < r_max)
    corrs = {}
    for angle in [30, 60, 90, 120, 150]:
        rotated = ndimage_rotate(autocorr, angle, reshape=False, order=1)
        corrs[angle] = np.corrcoef(autocorr[ring], rotated[ring])[0, 1] if ring.sum() > 10 else 0.0
    gridness = min(corrs[60], corrs[120]) - max(corrs[30], corrs[90], corrs[150])
    return gridness, autocorr, corrs

# Compute gridness for all neurons on (z_pos, angle) plane
n_g_list = cfg['n_g']
all_gridness = []
for f in range(n_f):
    module_scores = []
    for neuron in range(n_g_list[f]):
        rm, _ = compute_firing_map(grid_g_modules[f][:, neuron], grid_raw[:, 0], grid_raw[:, 1])
        gs, _, _ = compute_gridness(rm)
        module_scores.append(gs)
    all_gridness.append(module_scores)

# Plot gridness distribution + autocorrelation of best neuron per module
fig, axes = plt.subplots(2, n_f, figsize=(4 * n_f, 8))

for f in range(n_f):
    scores = all_gridness[f]
    best_idx = int(np.argmax(scores))

    ax = axes[0, f]
    ax.bar(range(len(scores)), sorted(scores, reverse=True), color='steelblue', alpha=0.7)
    ax.axhline(y=0.37, color='red', linestyle='--', alpha=0.5, label='Grid threshold')
    ax.set_title(f'Module {f} (alpha={cfg["f_initial"][f]:.2f})\nbest={scores[best_idx]:.3f}')
    ax.set_xlabel('Neuron (sorted)')
    ax.set_ylabel('Gridness')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.2)

    ax = axes[1, f]
    rm, _ = compute_firing_map(grid_g_modules[f][:, best_idx], grid_raw[:, 0], grid_raw[:, 1])
    _, autocorr, corrs = compute_gridness(rm)
    im = ax.imshow(autocorr, cmap='RdBu_r', vmin=-1, vmax=1, origin='lower')
    corr_str = ' '.join(f'{a}:{c:.2f}' for a, c in sorted(corrs.items()))
    ax.set_title(f'Neuron {best_idx} autocorrelation\n{corr_str}', fontsize=8)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Gridness Analysis\n(Periodic structure in spatial autocorrelation; threshold 0.37 from neuroscience)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

# Summary
print('Gridness scores per module:')
for f in range(n_f):
    s = all_gridness[f]
    n_grid = sum(1 for x in s if x > 0.37)
    print(f'  Module {f} (alpha={cfg["f_initial"][f]:.2f}): max={max(s):.3f}, mean={np.mean(s):.3f}, '
          f'grid cells (>{0.37}): {n_grid}/{len(s)}')

### 9. Adaptation Curves
How quickly does the TEM adapt to novel environments? We measure observation prediction MSE at each step within an episode across different environment types: default (no randomization), in-distribution, OOD extrapolation, and extreme single-parameter changes.

In [ ]:
def run_adaptation_episode(model, env_cfg, normalizer_fn, device, seed, max_steps=200):
    """Run one episode step-by-step, returning per-step MSE (recon + pred) and EMA."""
    cfg = model.cfg
    n_f = cfg['n_f']
    env = DomainRandomizedHopper(env_cfg, seed=seed)
    obs_raw, _ = env.reset()

    M = [model._init_episodic_buffer(1), model._init_episodic_buffer(1)]
    g_inf = [model.g_init[f].detach().unsqueeze(0).clone() for f in range(n_f)]
    x_inf = [torch.zeros(1, cfg['n_x_f'][f], device=device) for f in range(n_f)]
    transition_err_ema = None
    a_prev = None

    mses_recon, mses_pred, ema_vals = [], [], []
    for step in range(max_steps):
        obs_tensor = torch.tensor(normalizer_fn(obs_raw), dtype=torch.float, device=device).unsqueeze(0)
        with torch.no_grad():
            g_gen, g_gen_sigma = model._gen_g(a_prev, g_inf, transition_err_ema)
            x_f, g_new, p_inf_x, p_inf = model._inference(obs_tensor, M, x_inf, (g_gen, g_gen_sigma))
            x_gen, p_gen = model._generative(M, p_inf, g_new, g_gen)

            mses_recon.append(torch.mean((x_gen[0][0] - obs_tensor) ** 2).cpu().item())
            mses_pred.append(torch.mean((x_gen[2][0] - obs_tensor) ** 2).cpu().item())

            # EMA tracks transition prediction error (|x_pred - x_obs|)
            x_pred_mu, _ = model._gen_x(model._gen_p(g_gen, M[0]))
            obs_err = torch.mean(torch.abs(x_pred_mu - obs_tensor), dim=-1, keepdim=True)
            terr = [obs_err.expand(-1, cfg['n_g'][f]) for f in range(n_f)]
            if transition_err_ema is not None:
                decay = cfg.get('transition_err_ema_decay', 0.95)
                new_ema = [decay * transition_err_ema[f] + (1 - decay) * terr[f] for f in range(n_f)]
            else:
                new_ema = terr
            ema_vals.append(np.mean([new_ema[f].mean().cpu().item() for f in range(n_f)]))

            M_new = [model._episodic_store(M[0], torch.cat(p_inf, dim=1), torch.cat(p_gen, dim=1))]
            M_new.append(model._episodic_store(M[1], torch.cat(p_inf, dim=1), torch.cat(p_inf_x, dim=1), do_hierarchical=False))

            action = env.env.action_space.sample()
            a_prev = torch.tensor(action, dtype=torch.float, device=device).unsqueeze(0)
            obs_raw, _, terminated, truncated, _ = env.step(action)
            if terminated or truncated:
                break
            M, g_inf, x_inf, transition_err_ema = M_new, g_new, x_f, new_ema
    env.close()
    return np.array(mses_recon), np.array(mses_pred), np.array(ema_vals)

# Define test environment configs
env_configs = {}

cfg_default = cfg.copy()
cfg_default['randomize'] = False
cfg_default['gravity_range'] = None
cfg_default['terminate_when_unhealthy'] = False
env_configs['Default'] = cfg_default

cfg_id = cfg.copy()
cfg_id['terminate_when_unhealthy'] = False
env_configs['In-Distribution'] = cfg_id

cfg_ood = cfg.copy()
cfg_ood['terminate_when_unhealthy'] = False
cfg_ood['gravity_range'] = cfg.get('eval_gravity_range', (2.5, 4.0))
cfg_ood['friction_range'] = cfg.get('eval_friction_range', (0.02, 0.1))
cfg_ood['mass_range'] = cfg.get('eval_mass_range', (4.0, 8.0))
env_configs['OOD Extrapolation'] = cfg_ood

cfg_extg = cfg.copy()
cfg_extg['terminate_when_unhealthy'] = False
cfg_extg['mass_range'] = (1.0, 1.0)
cfg_extg['damping_range'] = (1.0, 1.0)
cfg_extg['friction_range'] = (1.0, 1.0)
cfg_extg['gear_range'] = (1.0, 1.0)
cfg_extg['gravity_range'] = (3.0, 4.0)
env_configs['Extreme Gravity'] = cfg_extg

# Run episodes
n_eval_eps = 10
max_eval_steps = 200
adapt_results = {}

for name, ecfg in env_configs.items():
    print(f'Running {name}...')
    all_mse_recon, all_mse_pred, all_ema = [], [], []
    for ep in range(n_eval_eps):
        mr, mp, e = run_adaptation_episode(model, ecfg, normalize_obs, device, seed=3000+ep, max_steps=max_eval_steps)
        all_mse_recon.append(mr); all_mse_pred.append(mp); all_ema.append(e)
    max_len = max(len(m) for m in all_mse_recon)
    mse_recon_mat = np.full((n_eval_eps, max_len), np.nan)
    mse_pred_mat = np.full((n_eval_eps, max_len), np.nan)
    ema_mat = np.full((n_eval_eps, max_len), np.nan)
    for i in range(n_eval_eps):
        mse_recon_mat[i, :len(all_mse_recon[i])] = all_mse_recon[i]
        mse_pred_mat[i, :len(all_mse_pred[i])] = all_mse_pred[i]
        ema_mat[i, :len(all_ema[i])] = all_ema[i]
    adapt_results[name] = {
        'mean_mse_recon': np.nanmean(mse_recon_mat, axis=0),
        'mean_mse_pred': np.nanmean(mse_pred_mat, axis=0),
        'std_mse_pred': np.nanstd(mse_pred_mat, axis=0),
        'mean_ema': np.nanmean(ema_mat, axis=0),
    }
    r = adapt_results[name]
    print(f'  Pred MSE step 1: {r["mean_mse_pred"][0]:.4f}, step 50: {r["mean_mse_pred"][min(49,max_len-1)]:.4f}, '
          f'asymptotic: {np.nanmean(r["mean_mse_pred"][min(149,max_len-1):]):.4f}')

# Plot
colors_adapt = {'Default': '#2196F3', 'In-Distribution': '#4CAF50',
                'OOD Extrapolation': '#FF5722', 'Extreme Gravity': '#9C27B0'}
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
for name, res in adapt_results.items():
    mean, std = res['mean_mse_recon'], res.get('std_mse_pred', np.zeros_like(res['mean_mse_recon']))
    ax.plot(np.arange(len(mean)), mean, color=colors_adapt[name], linewidth=2, label=name)
ax.set_xlabel('Step within episode')
ax.set_ylabel('Reconstruction MSE')
ax.set_title('Reconstruction MSE (from p_inf)')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
for name, res in adapt_results.items():
    mean, std = res['mean_mse_pred'], res['std_mse_pred']
    x = np.arange(len(mean))
    ax.plot(x, mean, color=colors_adapt[name], linewidth=2, label=name)
    ax.fill_between(x, mean - std, mean + std, color=colors_adapt[name], alpha=0.15)
ax.set_xlabel('Step within episode')
ax.set_ylabel('Prediction MSE')
ax.set_title('Transition Prediction MSE (from g_gen)')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[2]
for name, res in adapt_results.items():
    ax.plot(res['mean_ema'], color=colors_adapt[name], linewidth=2, label=name)
ax.set_xlabel('Step within episode')
ax.set_ylabel('Transition Error EMA')
ax.set_title('Transition Error EMA')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 10. Precision Weight Dynamics
The TEM infers abstract state g from three sources: **path integration** (transition model), **memory retrieval** (episodic buffer), and **direct observation**. Each source has learned uncertainty (sigma), and the final g is a precision-weighted (inverse-variance) combination.

Expected pattern in a novel environment:
- **Early steps**: observation dominates (transition model is wrong, buffer is empty)
- **Mid episode**: memory contribution rises as episodic buffer fills with associations
- **Late episode**: all three contribute as the transition model becomes accurate for this environment


In [ ]:
from tem_model import inv_var_weight, squared_error

def run_episode_with_precision(model, env_cfg, normalizer_fn, device, seed, max_steps=200):
    """Run one episode extracting per-step precision weights for each inference source."""
    cfg = model.cfg
    n_f = cfg['n_f']
    env = DomainRandomizedHopper(env_cfg, seed=seed)
    obs_raw, _ = env.reset()

    M = [model._init_episodic_buffer(1), model._init_episodic_buffer(1)]
    g_inf = [model.g_init[f].detach().unsqueeze(0).clone() for f in range(n_f)]
    x_inf = [torch.zeros(1, cfg['n_x_f'][f], device=device) for f in range(n_f)]
    transition_err_ema = None
    a_prev = None

    records = {'w_path': [], 'w_mem': [], 'w_obs': [],
               'sigma_path': [], 'sigma_mem': [], 'sigma_obs': [], 'mse': []}

    for step in range(max_steps):
        obs_tensor = torch.tensor(normalizer_fn(obs_raw), dtype=torch.float, device=device).unsqueeze(0)
        with torch.no_grad():
            # --- transition ---
            g_gen, sigma_g_path = model._gen_g(a_prev, g_inf, transition_err_ema)

            # --- encode + filter ---
            x_c = model.encoder(obs_tensor)
            x_f = model._temporal_filter(x_inf, x_c)
            x_ = model._x2x_(x_f)
            p_x = model._episodic_retrieve(x_, M[1], cfg['p_retrieve_mask_inf'])

            # --- memory-derived g and sigma ---
            g_down = [torch.matmul(p_x[f], cfg['W_repeat'][f].t()) for f in range(n_f)]
            mu_g_mem = model.MLP_mu_g_mem(g_down)
            x_hat_mu, _ = model._gen_x(p_x)
            err = squared_error(obs_tensor, x_hat_mu)
            sigma_in = [torch.cat([torch.sum(g**2, dim=1, keepdim=True),
                                   err.unsqueeze(1)], dim=1) for g in mu_g_mem]
            mu_g_mem = [torch.tanh(g) for g in mu_g_mem]
            sigma_g_mem_raw = model.MLP_sigma_g_mem(sigma_in)
            sigma_g_mem = [sigma_g_mem_raw[f] + cfg['p2g_scale_offset'] * cfg['p2g_sig_val']
                           + cfg.get('sigma_g_mem_floor', 0.35)
                           for f in range(n_f)]

            # --- observation-derived g and sigma ---
            mu_g_obs = [torch.tanh(g) for g in model.MLP_g_obs(x_f)]
            sigma_g_obs = [torch.exp(model.logsig_g_obs[f]).unsqueeze(0).expand_as(g_gen[f])
                           + cfg.get('sigma_g_obs_floor', 0.2)
                           for f in range(n_f)]

            # --- precision weights ---
            wp, wm, wo = [], [], []
            for f in range(n_f):
                iv_p = 1.0 / (sigma_g_path[f]**2 + 1e-8)
                iv_m = 1.0 / (sigma_g_mem[f]**2 + 1e-8)
                iv_o = 1.0 / (sigma_g_obs[f]**2 + 1e-8)
                total = iv_p + iv_m + iv_o
                wp.append((iv_p / total).mean().cpu().item())
                wm.append((iv_m / total).mean().cpu().item())
                wo.append((iv_o / total).mean().cpu().item())
            records['w_path'].append(np.mean(wp))
            records['w_mem'].append(np.mean(wm))
            records['w_obs'].append(np.mean(wo))
            records['sigma_path'].append(np.mean([sigma_g_path[f].mean().cpu().item() for f in range(n_f)]))
            records['sigma_mem'].append(np.mean([sigma_g_mem[f].mean().cpu().item() for f in range(n_f)]))
            records['sigma_obs'].append(np.mean([sigma_g_obs[f].mean().cpu().item() for f in range(n_f)]))

            # --- full inference for g, p, memory update ---
            g_new = []
            for f in range(n_f):
                mu, _ = inv_var_weight(
                    [g_gen[f], mu_g_mem[f], mu_g_obs[f]],
                    [sigma_g_path[f], sigma_g_mem[f], sigma_g_obs[f]])
                g_new.append(mu)
            g_ = model._g2g_(g_new)
            p_inf = model._inf_p(x_, g_)
            x_gen, p_gen = model._generative(M, p_inf, g_new, g_gen)
            records['mse'].append(torch.mean((x_gen[0][0] - obs_tensor)**2).cpu().item())

            M_new = [model._episodic_store(M[0], torch.cat(p_inf, dim=1), torch.cat(p_gen, dim=1))]
            M_new.append(model._episodic_store(M[1], torch.cat(p_inf, dim=1),
                                        torch.cat(p_x, dim=1), do_hierarchical=False))

            # EMA tracks transition prediction error
            x_pred_mu, _ = model._gen_x(model._gen_p(g_gen, M[0]))
            obs_err = torch.mean(torch.abs(x_pred_mu - obs_tensor), dim=-1, keepdim=True)
            terr = [obs_err.expand(-1, cfg['n_g'][f]) for f in range(n_f)]
            if transition_err_ema is not None:
                decay = cfg.get('transition_err_ema_decay', 0.95)
                new_ema = [decay * transition_err_ema[f] + (1-decay) * terr[f] for f in range(n_f)]
            else:
                new_ema = terr

            action = env.env.action_space.sample()
            a_prev = torch.tensor(action, dtype=torch.float, device=device).unsqueeze(0)
            obs_raw, _, terminated, truncated, _ = env.step(action)
            if terminated or truncated:
                break
            M, g_inf, x_inf, transition_err_ema = M_new, g_new, x_f, new_ema
    env.close()
    return {k: np.array(v) for k, v in records.items()}

# Run on two contrasting environments
print('Collecting precision weight dynamics...')
prec_default = run_episode_with_precision(model, cfg_default, normalize_obs, device, seed=4000)
prec_ood = run_episode_with_precision(model, cfg_ood, normalize_obs, device, seed=4001)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for col, (label, prec) in enumerate([('Default Hopper', prec_default), ('OOD Extrapolation', prec_ood)]):
    ax = axes[0, col]
    ax.stackplot(range(len(prec['w_path'])),
                 prec['w_path'], prec['w_mem'], prec['w_obs'],
                 labels=['Path integration', 'Memory', 'Observation'],
                 colors=['#2196F3', '#4CAF50', '#FF9800'], alpha=0.8)
    ax.set_ylabel('Precision Weight')
    ax.set_xlabel('Step')
    ax.set_title(f'{label}: Inference Source Weights')
    ax.set_ylim(0, 1)
    ax.legend(loc='right', fontsize=8)
    ax.grid(True, alpha=0.2)

    ax = axes[1, col]
    ax.semilogy(prec['sigma_path'], label='sigma path', color='#2196F3', linewidth=1.5)
    ax.semilogy(prec['sigma_mem'], label='sigma memory', color='#4CAF50', linewidth=1.5)
    ax.semilogy(prec['sigma_obs'], label='sigma observation', color='#FF9800', linewidth=1.5)
    ax.set_ylabel('Sigma (log scale)')
    ax.set_xlabel('Step')
    ax.set_title(f'{label}: Uncertainty per Source')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

plt.suptitle('Precision-Weighted Inference Dynamics', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


### 11. Full Evaluation with Ablations
Run the evaluation script to get comprehensive adaptation metrics and ablation comparisons (no EMA, no memory).

In [ ]:
!python evaluate_adaptation.py --model-dir {model_dir} --n-episodes 20 --ablations --mid-episode

### 12. Mid-Episode Physics Changes
The hardest test for the adaptation mechanism: physics change **without an episode reset**.
The episodic memory buffer and transition error EMA from the old environment are still active.
The system must detect the mismatch and re-adapt using the EMA → sigma_g_path pathway.

Expected pattern:
- **Before change**: low MSE, low EMA (adapted to initial physics)
- **At change**: MSE and EMA spike (transition model suddenly wrong, stale memory misleads)
- **Recovery**: EMA drives sigma_g_path up → trust shifts to observations → new episodic entries overwrite old ones via FIFO → MSE drops

In [ ]:
# Run mid-episode physics change experiment
change_step = 100  # step at which physics change
n_mid_eps = 10
mid_max_steps = 300

mid_changes = {
    'Default → Heavy Gravity (3x)': {'gravity_range': (3.0, 3.0), 'mass_range': (1.0, 1.0),
        'damping_range': (1.0, 1.0), 'friction_range': (1.0, 1.0), 'gear_range': (1.0, 1.0)},
    'Default → Low Friction (0.05x)': {'friction_range': (0.05, 0.05), 'mass_range': (1.0, 1.0),
        'damping_range': (1.0, 1.0), 'gravity_range': (1.0, 1.0), 'gear_range': (1.0, 1.0)},
    'Default → All Extreme': {'mass_range': (4.0, 4.0), 'gravity_range': (2.5, 2.5),
        'friction_range': (0.1, 0.1), 'damping_range': (3.0, 3.0), 'gear_range': (0.3, 0.3)},
}

# Start config: default physics
cfg_start = cfg.copy()
cfg_start['randomize'] = False
cfg_start['gravity_range'] = None
cfg_start['terminate_when_unhealthy'] = False

mid_results = {}
for name, override in mid_changes.items():
    print(f'Running {name}...')
    all_mse_recon, all_mse_pred, all_ema = [], [], []
    for ep in range(n_mid_eps):
        env = DomainRandomizedHopper(cfg_start, seed=6000 + ep)
        obs_raw, _ = env.reset()

        M = [model._init_episodic_buffer(1), model._init_episodic_buffer(1)]
        g_inf_s = [model.g_init[f].detach().unsqueeze(0).clone() for f in range(cfg['n_f'])]
        x_inf_s = [torch.zeros(1, cfg['n_x_f'][f], device=device) for f in range(cfg['n_f'])]
        terr_ema = None
        a_prev = None
        mses_recon, mses_pred, emas = [], [], []

        for step in range(mid_max_steps):
            if step == change_step:
                env.change_physics({**override, 'randomize': True})

            obs_tensor = torch.tensor(normalize_obs(obs_raw), dtype=torch.float, device=device).unsqueeze(0)
            with torch.no_grad():
                g_gen, g_sig = model._gen_g(a_prev, g_inf_s, terr_ema)
                x_f, g_new, p_inf_x, p_inf = model._inference(obs_tensor, M, x_inf_s, (g_gen, g_sig))
                x_gen, p_gen = model._generative(M, p_inf, g_new, g_gen)
                mses_recon.append(torch.mean((x_gen[0][0] - obs_tensor)**2).cpu().item())
                mses_pred.append(torch.mean((x_gen[2][0] - obs_tensor)**2).cpu().item())

                M_new = [model._episodic_store(M[0], torch.cat(p_inf, dim=1), torch.cat(p_gen, dim=1))]
                M_new.append(model._episodic_store(M[1], torch.cat(p_inf, dim=1), torch.cat(p_inf_x, dim=1), do_hierarchical=False))

                # EMA tracks transition prediction error (|x_pred - x_obs|)
                x_pred_mu, _ = model._gen_x(model._gen_p(g_gen, M[0]))
                obs_err = torch.mean(torch.abs(x_pred_mu - obs_tensor), dim=-1, keepdim=True)
                terr = [obs_err.expand(-1, cfg['n_g'][f]) for f in range(cfg['n_f'])]
                if terr_ema is not None:
                    decay = cfg.get('transition_err_ema_decay', 0.95)
                    new_ema = [decay * terr_ema[f] + (1-decay) * terr[f] for f in range(cfg['n_f'])]
                else:
                    new_ema = terr
                emas.append(np.mean([new_ema[f].mean().cpu().item() for f in range(cfg['n_f'])]))

                action = env.env.action_space.sample()
                a_prev = torch.tensor(action, dtype=torch.float, device=device).unsqueeze(0)
                obs_raw, _, terminated, truncated, _ = env.step(action)
                if terminated or truncated:
                    break
                M, g_inf_s, x_inf_s, terr_ema = M_new, g_new, x_f, new_ema
        env.close()
        all_mse_recon.append(np.array(mses_recon))
        all_mse_pred.append(np.array(mses_pred))
        all_ema.append(np.array(emas))

    max_len = max(len(m) for m in all_mse_recon)
    mse_recon_mat = np.full((n_mid_eps, max_len), np.nan)
    mse_pred_mat = np.full((n_mid_eps, max_len), np.nan)
    ema_mat = np.full((n_mid_eps, max_len), np.nan)
    for i in range(n_mid_eps):
        mse_recon_mat[i, :len(all_mse_recon[i])] = all_mse_recon[i]
        mse_pred_mat[i, :len(all_mse_pred[i])] = all_mse_pred[i]
        ema_mat[i, :len(all_ema[i])] = all_ema[i]
    mid_results[name] = {
        'mean_mse_recon': np.nanmean(mse_recon_mat, axis=0),
        'mean_mse_pred': np.nanmean(mse_pred_mat, axis=0),
        'mean_ema': np.nanmean(ema_mat, axis=0),
    }

# Plot
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
colors_mid = list(plt.cm.Set1.colors)

ax = axes[0]
for i, (name, res) in enumerate(mid_results.items()):
    ax.plot(res['mean_mse_recon'], color=colors_mid[i], linewidth=2, label=name)
ax.axvline(x=change_step, color='black', linestyle='--', linewidth=2, alpha=0.7, label='Physics change')
ax.set_ylabel('Reconstruction MSE')
ax.set_title('Reconstruction MSE (from p_inf — sees observation)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
for i, (name, res) in enumerate(mid_results.items()):
    ax.plot(res['mean_mse_pred'], color=colors_mid[i], linewidth=2, label=name)
ax.axvline(x=change_step, color='black', linestyle='--', linewidth=2, alpha=0.7, label='Physics change')
ax.set_ylabel('Prediction MSE')
ax.set_title('Transition Prediction MSE (from g_gen — blind to current observation)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[2]
for i, (name, res) in enumerate(mid_results.items()):
    ax.plot(res['mean_ema'], color=colors_mid[i], linewidth=2, label=name)
ax.axvline(x=change_step, color='black', linestyle='--', linewidth=2, alpha=0.7, label='Physics change')
ax.set_ylabel('Transition Error EMA')
ax.set_xlabel('Step')
ax.set_title('Transition Error EMA detects the regularity change')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary
for name, res in mid_results.items():
    m = res['mean_mse_pred']
    pre = np.nanmean(m[max(0,change_step-20):change_step])
    spike = np.nanmax(m[change_step:min(change_step+20, len(m))]) if change_step < len(m) else float('nan')
    recov = np.nanmean(m[change_step+50:change_step+100]) if change_step+50 < len(m) else float('nan')
    print(f'{name}: pre={pre:.4f} → spike={spike:.4f} ({spike/pre:.1f}x) → recovery={recov:.4f}')


In [ ]:
# Download results
import glob, os
run_dirs = sorted(glob.glob('runs/*/models'))
if run_dirs:
    latest = os.path.dirname(run_dirs[-1])
    print(f"Latest run: {latest}")
    !zip -r results.zip {latest}
    from google.colab import files
    files.download('results.zip')
else:
    print("No runs found yet.")